In [1]:
# Two-Tower Recommendation (InfoNCE) usando MovieLens 100k
# Carga datos con tensorflow_datasets, entrena en PyTorch.
# Importado librerias
#importamos tfds porque facilita obtener MovieLens ya limpio.
import tensorflow_datasets as tfds
import numpy as np
#torch, nn, optim, DataLoader son la base para construir, entrenar y alimentar modelos PyTorch.
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader


In [2]:

# 1) Cargar MovieLens 100k (ratings + movies)
# Usamos tfds por simplicidad: devuelve tf.data.Dataset con campos útiles.
ratings_ds = tfds.load("movielens/100k-ratings", split="train") # interacciones entre los usuarios y las peliculas
movies_ds  = tfds.load("movielens/100k-movies", split="train") # lista de peliculas para los usarios

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

Generating splits...:   0%|          | 0/1 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

...

Extraction completed...: 0 file [00:00, ? file/s]

Generating splits...:   0%|          | 0/1 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/movielens/100k-movies/incomplete.I8UJ8X_0.1.1/movielens-train.tfrecord*...…

Dataset movielens downloaded and prepared to /root/tensorflow_datasets/movielens/100k-movies/0.1.1. Subsequent calls will reuse this data.


In [3]:
# Extraemos pares (user_id, movie_title).
user_list = []
movie_list = []
pairs = []  #

# Iteramos sobre ratings para convertir cada registro en string con numpy.decode.
# Esto se realiza de esta manera por Pytorch no opera con tf.tensor directamento por eso lo convertimos a numpy para crear el conjunto de datos de tipo pytorch.
for r in ratings_ds:
    u = r["user_id"].numpy().decode("utf-8")
    m = r["movie_title"].numpy().decode("utf-8")
    user_list.append(u)
    movie_list.append(m)
    pairs.append((u, m))


# Obtenemos vocabularios únicos ordenados ya que necesitamos un indice por usuario y pelicula para el proceso de embeddging.
unique_users = sorted(list(set(user_list)))
unique_movies = sorted(list(set(movie_list)))

num_users = len(unique_users)
num_movies = len(unique_movies)
print(f"Usuarios únicos: {num_users}, Películas únicas: {num_movies}")

# Mappings string <-> index para poder construir los embedding.
user_to_idx = {u: i for i, u in enumerate(unique_users)}
movie_to_idx = {m: i for i, m in enumerate(unique_movies)}
idx_to_movie = {i: m for m, i in movie_to_idx.items()}

# Convertir pares a índices numpy o numericos para el mismo proceso.
user_indices = np.array([user_to_idx[u] for u, _ in pairs], dtype=np.int64)
movie_indices = np.array([movie_to_idx[m] for _, m in pairs], dtype=np.int64)


Usuarios únicos: 943, Películas únicas: 1664


In [4]:
# -----------------------------
# 2) PyTorch Dataset y DataLoader
# -----------------------------
class InteractionsDataset(Dataset):
    def __init__(self, users_np, movies_np):
        self.users = torch.from_numpy(users_np).long() # Esta linea codigo convierte los arrays en de numpya una tensor de pytorch para poder utilizarlo y se asegura los datos sean enteros.
        self.movies = torch.from_numpy(movies_np).long()

    def __len__(self):
        return self.users.shape[0]

    def __getitem__(self, idx):
        return {
            "user": self.users[idx],
            "movie": self.movies[idx]
        }

# Hiperparámetros
batch_size = 2048
dataset = InteractionsDataset(user_indices, movie_indices)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, pin_memory=True, drop_last=True)
#batch_size: número de ejemplos por paso,controla estabilidad y nivel de negatives in-batch.
#shuffle=True: mezcla las interacciones cada epoch, para mejor generalización.
#pin_memory=True: mejora la transferencia a GPU si existe.
#drop_last=True: asegura batch completos, importante para el loss in-batch.

In [5]:
# 3) Modelo Two-Tower (PyTorch)
# -----------------------------
class UserTower(nn.Module):
    def __init__(self, num_users, emb_dim=64, hidden_dim=128, dropout=0.2): # crea una tabla de embeddings de dimensión emb_dim 64. Cada fila corresponde a un usuario.
        super().__init__()
        self.emb = nn.Embedding(num_users, emb_dim) # Crea vectores para los usuarios los cuales son la base de la representación.
        # Dropout
        self.dropout = nn.Dropout(dropout) # Se utiliza ara evitar el sobreajuste
        self.mlp = nn.Sequential(  # Esta linea permite que el modelo combine dimensiones, aprenda relaciones no lineales y adapte espaciones usuarios pelicuales
            nn.Linear(emb_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, emb_dim)
        )

    def forward(self, user_idx): #normaliza vector L2. Ventaja: estabiliza producto punto y hace que la similitud sea coseno (proporcional al producto punto si están normalizados).
        # user_idx: tensor (B,)
        x = self.emb(user_idx)
        x = self.dropout(x) # Dropout en embeddings
        x = self.mlp(x)    # Proyección no lineal
        x = nn.functional.normalize(x, p=2, dim=1)
        return x

class MovieTower(nn.Module): # los comentario anterior aplicando igual.
    def __init__(self, num_movies, emb_dim=64, hidden_dim=128, dropout=0.2):
        super().__init__()
        self.emb = nn.Embedding(num_movies, emb_dim)
        self.dropout = nn.Dropout(dropout)
        self.mlp = nn.Sequential(
            nn.Linear(emb_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, emb_dim)
        )

    def forward(self, movie_idx):
        x = self.emb(movie_idx)
        x = self.dropout(x)
        x = self.mlp(x)
        x = nn.functional.normalize(x, p=2, dim=1)
        return x

class TwoTowerModel(nn.Module):
    def __init__(self, num_users, num_movies, emb_dim=64, temperature=0.07):
        super().__init__()
        self.user_tower = UserTower(num_users, emb_dim)
        self.movie_tower = MovieTower(num_movies, emb_dim)
        self.temperature = temperature

    def forward(self, user_idx, movie_idx):
        u = self.user_tower(user_idx)
        v = self.movie_tower(movie_idx)
        return u, v

    def compute_logits(self, u, v): # Calcula la similitud de los usuarios y peliculas
        return torch.matmul(u, v.t()) / self.temperature # controla la suavidad de la distribución de probabilidades y la dificultad de la tarea contrastiva.

# Instanciar modelo
emb_dim = 64
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = TwoTowerModel(num_users, num_movies, emb_dim=emb_dim).to(device)

In [6]:
# -----------------------------
# 4) Entrenamiento (InfoNCE in-batch)
# -----------------------------
optimizer = optim.Adam(model.parameters(), lr=3e-3,weight_decay=1e-5) # definiendo el optimizador.
loss_fn = nn.CrossEntropyLoss() # inicializando la fundacion de perdida.

n_epochs = 50
""" Este es el bloque principal de entrenamient del modelo dos torres. El objetvo es iterar a traves de los datos
    de entrenwe varias veces para ajustar los parametros, osea los embeddings de usuarios y peliculas para identificar
    aquellos pares positivos y hacer recomendaciones"""
for epoch in range(1, n_epochs + 1):
    model.train() # pone el modelo en esado de entrenamiento.
    running_loss = 0.0  # inicializa una variable para acumular la pérdida total dentro de cada época.
    for batch in dataloader:
        user_idx = batch["user"].to(device) # Extrae los índices de usuario del lote actual y los mueve al dispositivo de cómputo ('cuda' si hay GPU disponible, o 'cpu' en caso contrario).
        movie_idx = batch["movie"].to(device)

        optimizer.zero_grad() # Pone los grandientes acumulados de la iteración anterior en cero. Esto es debido a que pytorch acumular los gradientes por defecto y si no se reinician se sumarias y las actualizaciones serian incorrectas.
        u_emb, v_emb = model(user_idx, movie_idx) #El modelo calcula y devuelve los embeddings de usuario (u_emb) y película (v_emb) para este lote.

        logits = model.compute_logits(u_emb, v_emb) # Utiliza los embeddings para calcular la similitud entre los emb usuarios y peliculas del lote.
        labels = torch.arange(logits.size(0), device=device) # Crea las etiquetas positivas para la función de perdida.

        loss = loss_fn(logits, labels) # Calcula la perdida para el lote actual utilizando la función de perdida.
        loss.backward() # Realiza el paso hacia atrás (backward pass) o retropropagación. Calcula los gradientes de la pérdida con respecto a todos los parámetros aprendibles del modelo
        optimizer.step() # Actualiza los parámetros del modelo utilizando los gradientes calculados en el paso anterior y el algoritmo del optimizador (Adam en este caso). El optimizador ajusta los embeddings de usuario y película para reducir la pérdida.

        running_loss += loss.item() # Suma la pérdida del lote actual a running_loss.item() se usa para obtener el valor escalar de la pérdida de un tensor de PyTorch.

    avg_loss = running_loss / len(dataloader) # Al final de cada época, calcula la pérdida promedio dividiendo la pérdida acumulada por el número total de lotes en el dataloader.
    print(f"Epoch {epoch}/{n_epochs} - Loss: {avg_loss:.4f}") # imprime la pérdida promedio para la época actual, lo que te permite monitorear el progreso del entrenamiento.

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch 1/50 - Loss: 7.6960
Epoch 2/50 - Loss: 7.6256
Epoch 3/50 - Loss: 7.6237
Epoch 4/50 - Loss: 7.6224
Epoch 5/50 - Loss: 7.6198
Epoch 6/50 - Loss: 7.6115
Epoch 7/50 - Loss: 7.5684
Epoch 8/50 - Loss: 7.4833
Epoch 9/50 - Loss: 7.4115
Epoch 10/50 - Loss: 7.3586
Epoch 11/50 - Loss: 7.3247
Epoch 12/50 - Loss: 7.2979
Epoch 13/50 - Loss: 7.2793
Epoch 14/50 - Loss: 7.2621
Epoch 15/50 - Loss: 7.2512
Epoch 16/50 - Loss: 7.2400
Epoch 17/50 - Loss: 7.2309
Epoch 18/50 - Loss: 7.2219
Epoch 19/50 - Loss: 7.2115
Epoch 20/50 - Loss: 7.2015
Epoch 21/50 - Loss: 7.1927
Epoch 22/50 - Loss: 7.1831
Epoch 23/50 - Loss: 7.1738
Epoch 24/50 - Loss: 7.1650
Epoch 25/50 - Loss: 7.1551
Epoch 26/50 - Loss: 7.1500
Epoch 27/50 - Loss: 7.1404
Epoch 28/50 - Loss: 7.1359
Epoch 29/50 - Loss: 7.1277
Epoch 30/50 - Loss: 7.1192
Epoch 31/50 - Loss: 7.1156
Epoch 32/50 - Loss: 7.1094
Epoch 33/50 - Loss: 7.1044
Epoch 34/50 - Loss: 7.1008
Epoch 35/50 - Loss: 7.0949
Epoch 36/50 - Loss: 7.0900
Epoch 37/50 - Loss: 7.0836
Epoch 38/5

In [8]:
# -----------------------------
# 5) Construir índice de películas y función de recomendación
# -----------------------------

# En este bloque de codigo evaluamos el modelo y se precalcular los embeddings para futuras recomendaciones.
model.eval() # Evalua el modelo.
with torch.no_grad(): # esto es un context manager que desactiva el calculo de gradiente ya que no es necesario actualizar los pesos en esta etapa.
    # embeddings para todas las películas (num_movies, emb_dim)
    movie_idx_all = torch.arange(num_movies, dtype=torch.long, device=device) # Crea un tensor que contiene todos los índices posibles de las películas, desde 0 hasta num_movies - 1.
    movie_emb_all = model.movie_tower(movie_idx_all)  # Pasa el tensor con todos los índices de película a través de la movie_tower del modelo. Esto genera los embeddings para todas las películas de una sola vez.

# Esta función es la que permite obtener las recomendaciones de peliculas para un usuarios dado, utilizando el modelo Two-Tower.
def recommend_for_user(user_str, k=10):
    if user_str not in user_to_idx: # Verifica si el user_str proporcionado existe en el vocabulario de usuarios que el modelo conoce (user_to_idx).
        raise ValueError("Usuario no está en el vocabulario")
    u_idx = torch.tensor([user_to_idx[user_str]], dtype=torch.long, device=device) # onvierte el ID de usuario en una entrada numérica (índice) compatible con PyTorch y lo prepara para el modelo.
    with torch.no_grad():
        u_emb = model.user_tower(u_idx)  # Obtiene el vector de embedding del usuario para el user_str dado.
        scores = torch.matmul(movie_emb_all, u_emb.t()).squeeze(1)  #Calcula la similitud entre el usuario dado y todas las películas en tu dataset.
        topk = torch.topk(scores, k=k) # Encuentra las k películas con los puntajes de similitud más altos.
        indices = topk.indices.cpu().numpy().tolist() # Convierte los resultados de PyTorch (topk.indices, topk.values) a listas de Python estándar para facilitar su procesamiento y visualización.
        values = topk.values.cpu().numpy().tolist()
    return [(idx_to_movie[i], float(values[j])) for j, i in enumerate(indices)] # Formatea las recomendaciones en una lista de tuplas (nombre_pelicula, puntuación_similitud).

# Ejemplo de uso
example_user = unique_users[90]
print("Recomendaciones para", example_user, recommend_for_user(example_user, k=5))

Recomendaciones para 180 [('Reality Bites (1994)', 0.5036225318908691), ('Threesome (1994)', 0.4659392237663269), ('Better Off Dead... (1985)', 0.4645088315010071), ('Before Sunrise (1995)', 0.4638763964176178), ('Sirens (1994)', 0.4600215256214142)]


## **Explicación:**

* La razón por la cual seleccioné el sistema de recomendación basado en embeddings o en modelos y no el tradicional basado en items, debido a que es más moderno y potente, capaz de captura relaciones más complejas mediante redes neuronales. Quiero adquirir más experiencias este tipo de sistemas de recomendación. Además entiendo que es mucho más eficiente y productivo sus recomendaciones tendrán mmejor precisión y con mucha profundidad por caprtuar las relaciones complejas entre los usuarios y las peliculas.

* Pero cual es la diferencia entre estos dos tipos de enfoque de filtrado colaborativo?, el basado en modelos aprende a representar, una de las especialidades de las redes neuronales profundas, los usuarios y items para predecir preferencias más comlejas de los usuarios. El segundo, basado en items o usuarios calcula la similitudes desde la matriz de iteraciones, de manera diferente que el primero, que encuentra alas similitudes a traves de los embeddings por sus cercanias.

* La función recommend_for_user ilustra el funcionamiento del sistema obteniendo el embedding de los usuarios (u_emb), luego calcula la similitud de este u_emb con todos los demas (movie_emb_all) y hace los precalculos. Finalmente selecciona las peliculas con mayor similitud.




## **Observaciones:**

* Estos resultados nos dicen que los usuarios y peliculas tienen alta similitud.
* Para obtener estos resultados se obtuvieron al modificar algunos parametros ya que nos arrojaron una similitud de coseno cerca 0.04.
* Aumentamos la cantidad de epoch de 5 a 50, esto con el objetivo de que el modelo interactua más con el conjunto de datos entrenamiento y pueda reconocer mejor los patrones de los usuarios.
* Agregamos capas dropout para el sobre ajuste.
* Agregamos capas mlp para que el modelo pudiera capturar relaciones complejas o no lineales.
* Normalizamos las recomendaciones para mejorar el rendimiento del modelo algunos embeddings no tegnas más pesos que otros al momento de realizar el calculo de coseno.
* A traves de lectura adicionales los valores que estan normalizados 50 y 70 en puntuaciones de coseno son fuertes y estables, dando entender el buen rendimiento del sistema en recomendación.
* Estos resultados nos dicen, ademas de los expuesto anteriormente, que el modelo esta agrupando muy bien y que esta capturando las preferencias de los usuarios muy bien.
* Adicional, estos valores nos indican preferencias consistentes con el historial y buena separación respecto de otras películas.



## Nota: utilice Pytorch por dos razones:
* Queria utilizar ese framework ya que tengo planes de dominarlo para futuros estudios y tenia curiosida por implementarlo.
* Las versiones de tensorflow y tensorflow_recommender estaban dando problemas por versiones diferentes, realicé varios intentos de cambios de versiones no dieron los resultados esperadon otra razón por la cual opter por utilizar Pytorch.